In [ ]:
from pathlib import Path
import os

OUTPUT_DIR = Path("../data/OutputDiar")
ASR_MODEL  = os.environ.get("SC_ASR_MODEL", "medium")
DEVICE     = "cuda"
DTYPE      = "float16"
MIN_SIL    = 1.5

ROLES_MAP = {}
USE_ALIGNED_FOR_ROLES = True

print("OUTPUT_DIR:", OUTPUT_DIR)
print("ASR_MODEL:", ASR_MODEL, "| DEVICE:", DEVICE, "| DTYPE:", DTYPE, "| MIN_SIL:", MIN_SIL)

In [ ]:
import torch, pandas as pd, numpy as np

def load_index(output_dir: Path):
    idx_path = output_dir / "diarization_index.csv"
    assert idx_path.exists(), f"Index not found at {idx_path}"
    idx = pd.read_csv(idx_path)
    assert {"csv_path","wav_path"}.issubset(idx.columns)
    return idx

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from faster_whisper import WhisperModel

_asr_model = None
def get_asr():
    global _asr_model
    if _asr_model is None:
        _asr_model = WhisperModel(ASR_MODEL, device=DEVICE, compute_type=DTYPE)
    return _asr_model

def transcribe_words_df(wav_path: Path) -> pd.DataFrame:
    model = get_asr()
    segments, info = model.transcribe(
        str(wav_path), beam_size=5, vad_filter=True, word_timestamps=True, condition_on_previous_text=True
    )
    rows = []
    for seg in segments:
        if seg.words:
            for w in seg.words:
                if w.start is None or w.end is None:
                    continue
                rows.append({"start": float(w.start), "end": float(w.end), "word": w.word})
    return pd.DataFrame(rows).sort_values("start").reset_index(drop=True)

In [ ]:
def infer_roles_from_aligned(diar_csv: Path):
    aligned = diar_csv.with_name(diar_csv.stem.replace("_diarization","_aligned_segments") + ".csv")
    if not aligned.exists():
        return {}
    df = pd.read_csv(aligned)
    if "role" not in df.columns or "speaker" not in df.columns:
        return {}
    tmp = df.groupby(["speaker","role"], as_index=False).size().sort_values(["speaker","size"], ascending=[True, False])
    roles = {}
    for spk in tmp["speaker"].unique():
        row = tmp[tmp["speaker"]==spk].iloc[0]
        roles[spk] = row["role"]
    return roles

def get_roles_map(diar_csv: Path):
    if ROLES_MAP:
        return ROLES_MAP
    if USE_ALIGNED_FOR_ROLES:
        rm = infer_roles_from_aligned(diar_csv)
        if rm:
            return rm
    return {}

In [ ]:
def words_in_window(words_df, start, end):
    if words_df is None or words_df.empty:
        return pd.DataFrame(columns=["start","end","word"])
    w = words_df[(words_df["start"] < end) & (words_df["end"] > start)].copy()
    if w.empty:
        return w
    w["start"] = w["start"].clip(lower=start)
    w["end"]   = w["end"].clip(upper=end)
    return w.sort_values("start").reset_index(drop=True)

def continuous_blocks_from_diar_exact(diar_df, words_df, min_sil=1.5, roles_map=None):
    roles_map = roles_map or {}
    diar = diar_df.sort_values("start").reset_index(drop=True)
    rows = []
    last_end = None
    for _, seg in diar.iterrows():
        s, e, spk = float(seg["start"]), float(seg["end"]), seg["speaker"]
        role = roles_map.get(spk, spk)
        if last_end is not None and (s - last_end) >= min_sil:
            rows.append({"start": last_end, "end": s, "dur": s-last_end, "type":"silence", "speaker":"", "role":"", "text":"", "n_words":0})
        ww = words_in_window(words_df, s, e)
        if ww.empty:
            rows.append({"start": s, "end": e, "dur": e-s, "type":"speech", "speaker": spk, "role": role, "text": "", "n_words":0})
            last_end = e
            continue
        
        if ww.iloc[0]["start"] - s >= min_sil:
            rows.append({"start": s, "end": ww.iloc[0]["start"], "dur": ww.iloc[0]["start"]-s, "type":"silence", "speaker":"", "role":"", "text":"", "n_words":0})
        
        chunk_start = ww.iloc[0]["start"]
        chunk_words = [ww.iloc[0]["word"]]
        for i in range(1, len(ww)):
            prev_end = ww.iloc[i-1]["end"]
            cur_start = ww.iloc[i]["start"]
            gap = cur_start - prev_end
            if gap >= min_sil:
                rows.append({"start": chunk_start, "end": prev_end, "dur": prev_end-chunk_start, "type":"speech", "speaker": spk, "role": role, "text": " ".join(chunk_words), "n_words": len(chunk_words)})
                rows.append({"start": prev_end, "end": cur_start, "dur": gap, "type":"silence", "speaker":"", "role":"", "text":"", "n_words":0})
                chunk_start = cur_start
                chunk_words = [ww.iloc[i]["word"]]
            else:
                chunk_words.append(ww.iloc[i]["word"])
        final_end = ww.iloc[-1]["end"]
        rows.append({"start": chunk_start, "end": final_end, "dur": final_end-chunk_start, "type":"speech", "speaker": spk, "role": role, "text": " ".join(chunk_words), "n_words": len(chunk_words)})
        
        if e - ww.iloc[-1]["end"] >= min_sil:
            rows.append({"start": ww.iloc[-1]["end"], "end": e, "dur": e-ww.iloc[-1]["end"], "type":"silence", "speaker":"", "role":"", "text":"", "n_words":0})
        last_end = e
    return pd.DataFrame(rows).sort_values("start").reset_index(drop=True)

In [ ]:
from tqdm import tqdm

def build_strict_continuous_files(output_dir: Path, min_sil=1.5, force=False):
    idx = load_index(output_dir)
    for _, row in tqdm(idx.iterrows(), total=len(idx)):
        diar_csv = output_dir / row["csv_path"]
        wav_path = output_dir / row["wav_path"]
        out_csv = diar_csv.with_name(diar_csv.stem.replace("_diarization","_continuous_timeline") + ".csv")
        if out_csv.exists() and not force:
            continue
        diar_df = pd.read_csv(diar_csv)
        words_df = transcribe_words_df(wav_path)
        roles_map = get_roles_map(diar_csv)
        continuous = continuous_blocks_from_diar_exact(diar_df, words_df, min_sil=min_sil, roles_map=roles_map)
        diar_total = (diar_df["end"] - diar_df["start"]).sum()
        speech_total = continuous.loc[continuous["type"]=="speech", "dur"].sum()
        print(out_csv.name, "speech_vs_diar_delta:", abs(diar_total-speech_total))
        continuous.to_csv(out_csv, index=False)
    print("Done.")

In [ ]:
build_strict_continuous_files(OUTPUT_DIR, min_sil=MIN_SIL, force=False)